In [21]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from dataclasses import dataclass
from sklearn.metrics import roc_auc_score, log_loss

In [ ]:
import polars as pl


shape: (2, 1)
┌─────┐
│ a   │
│ --- │
│ f64 │
╞═════╡
│ 1.2 │
│ 1.5 │
└─────┘


In [3]:
@dataclass
class Config:
    # 5-секундные тики -> 17280.
    ticks_per_day: int = 17280

    lookback_days: int = 30      # глубина истории для признаков
    horizon_days: int = 1        # длина окна, в котором ищем метки
    gap_days: int = 2            # зазор между train / valid / test
    n_users: int = 5000          # сколько пользователей берём на один снимок
    n_hist_cands: int = 100      # кандидатов из истории юзера (на юзера)
    n_pop_cands: int = 50        # глобально популярных кандидатов (одинаковые для всех)
    pos_threshold: float = 50.0  # played_ratio_pct >= 50 считаем позитивом (listen+)
    tau_days: tuple = (1.0, 7.0) # постоянные времени для затухающих счётчиков
    seed: int = 42

In [6]:
import polars as pl
from datasets import load_dataset

# Загрузка через Hugging Face datasets (возвращает dict с ключом 'train')
# Для 50M все файлы небольшие, можно загрузить в память
listens = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="listens.parquet")["train"]
likes = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")["train"]
dislikes = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="dislikes.parquet")["train"]
multi_event = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="multi_event.parquet")["train"]

# Конвертация в Polars DataFrame для скорости
df_listens = pl.from_arrow(listens.data.table)
df_likes = pl.from_arrow(likes.data.table)
df_dislikes = pl.from_arrow(dislikes.data.table)
df_multi = pl.from_arrow(multi_event.data.table)

In [7]:
print("Схема listens:", df_listens.schema)
print("Схема multi_event:", df_multi.schema)
print("Число строк в listens:", df_listens.height)
print("Число строк в likes:", df_likes.height)
print("Число строк в dislikes:", df_dislikes.height)

Схема listens: Schema({'uid': UInt32, 'timestamp': UInt32, 'item_id': UInt32, 'is_organic': UInt8, 'played_ratio_pct': UInt16, 'track_length_seconds': UInt32})
Схема multi_event: Schema({'uid': UInt32, 'timestamp': UInt32, 'item_id': UInt32, 'is_organic': UInt8, 'played_ratio_pct': UInt16, 'track_length_seconds': UInt32, 'event_type': String})
Число строк в listens: 46467212
Число строк в likes: 881456
Число строк в dislikes: 107776


In [17]:
df_listens.sample(5)

uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u16,u32
818200,5692240,4300728,1,0,240
136800,11202360,4683819,1,100,165
340400,11698565,6624375,0,100,190
566300,24876250,8071116,0,100,145
252600,12731860,1232814,1,0,205


In [18]:
df_likes.sample(5)

uid,timestamp,item_id,is_organic
u32,u32,u32,u8
971500,22965190,5646953,1
167000,24330080,8651525,0
236400,9465290,6247530,1
673000,9041660,7224068,1
554700,5420535,5597692,1


In [19]:
df_dislikes.sample(5)

uid,timestamp,item_id,is_organic
u32,u32,u32,u8
434400,12039655,6824734,0
284500,6979725,7443273,0
794900,6699670,8202269,1
343100,16874460,172078,1
862300,21707475,3706702,0


In [16]:
df_multi.sample(5)

uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
u32,u32,u32,u8,u16,u32,str
37700,8090860,8852914,1,1,215,"""listen"""
716700,726395,8229460,1,51,200,"""listen"""
373900,16548635,2363329,1,1,175,"""listen"""
579800,20467460,2833149,1,null,null,"""like"""
363800,6429445,5016149,0,100,240,"""listen"""


In [8]:
for name, df in [("listens", df_listens), ("likes", df_likes), ("dislikes", df_dislikes), ("multi_event", df_multi)]:
    null_counts = df.null_count()
    print(f"\nПропуски в {name}:")
    print(null_counts)


Пропуски в listens:
shape: (1, 6)
┌─────┬───────────┬─────────┬────────────┬──────────────────┬──────────────────────┐
│ uid ┆ timestamp ┆ item_id ┆ is_organic ┆ played_ratio_pct ┆ track_length_seconds │
│ --- ┆ ---       ┆ ---     ┆ ---        ┆ ---              ┆ ---                  │
│ u32 ┆ u32       ┆ u32     ┆ u32        ┆ u32              ┆ u32                  │
╞═════╪═══════════╪═════════╪════════════╪══════════════════╪══════════════════════╡
│ 0   ┆ 0         ┆ 0       ┆ 0          ┆ 0                ┆ 0                    │
└─────┴───────────┴─────────┴────────────┴──────────────────┴──────────────────────┘

Пропуски в likes:
shape: (1, 4)
┌─────┬───────────┬─────────┬────────────┐
│ uid ┆ timestamp ┆ item_id ┆ is_organic │
│ --- ┆ ---       ┆ ---     ┆ ---        │
│ u32 ┆ u32       ┆ u32     ┆ u32        │
╞═════╪═══════════╪═════════╪════════════╡
│ 0   ┆ 0         ┆ 0       ┆ 0          │
└─────┴───────────┴─────────┴────────────┘

Пропуски в dislikes:
shape: (1, 4)


In [ ]:
class SnapshotBuilder:
    """Строит обучающую таблицу для ОДНОЙ точки T."""

    def __init__(self, listens: pd.DataFrame, likes: pd.DataFrame, cfg: Config):
        self.cfg = cfg
        self.day = cfg.ticks_per_day
        self.rng = np.random.default_rng(cfg.seed)

        # Сортировка по времени: это условие для searchsorted
        self.listens = listens.sort_values("timestamp").reset_index(drop=True)
        self.likes = likes.sort_values("timestamp").reset_index(drop=True)

        # Массивы времени отдельно: бинарный поиск по numpy быстрый
        self._l_ts = self.listens["timestamp"].to_numpy()
        self._k_ts = self.likes["timestamp"].to_numpy()

    @staticmethod
    def _slice(df: pd.DataFrame, ts: np.ndarray, start: int, end: int) -> pd.DataFrame:
        """Строки с timestamp в [start, end). Правая граница НЕ включена -> нет утечки."""
        lo = np.searchsorted(ts, start, side="left")
        hi = np.searchsorted(ts, end, side="left")
        return df.iloc[lo:hi]
    def build(self, T: int) -> pd.DataFrame:
        c = self.cfg
        lb = c.lookback_days * self.day

        # 1) Разрезаем время: прошлое (для признаков) и будущее (для меток)
        hist = self._slice(self.listens, self._l_ts, T - lb, T).copy()
        fut = self._slice(self.listens, self._l_ts, T, T + c.horizon_days * self.day)
        likes_h = self._slice(self.likes, self._k_ts, T - lb, T)

        # 2) Базовые поля истории: возраст события в днях, клип процента, флаг listen+
        hist["age_days"] = (T - hist["timestamp"]) / self.day          # > 0, т.к. ts < T
        hist["ratio"] = hist["played_ratio_pct"].clip(0, 100)          # >100 бывает из-за перемоток
        hist["is_plus"] = (hist["ratio"] >= c.pos_threshold).astype(int)

        # 3) Выбираем пользователей и оставляем только их историю для user/pair признаков
        users = self._pick_users(hist, fut)
        hist_u = hist[hist["uid"].isin(users)]

        # 4) Признаки пары (юзер, трек) -> нужны и для кандидатов, и как фичи
        pair = self._pair_stats(hist_u)

        # 5) Кандидаты: история юзера + глобально популярные
        cands = self._candidates(pair, hist, users)

        # 6) Присоединяем признаки pair / user / item
        df = cands.merge(pair, on=["uid", "item_id"], how="left")
        df = df.merge(self._user_feats(hist_u), on="uid", how="left")
        df = df.merge(self._item_feats(hist, likes_h), on="item_id", how="left")

        # У кандидатов без истории пара-счётчики = 0 (а days_since_last остаётся NaN: "никогда")
        zero_cols = [col for col in pair.columns if col.startswith(("pair_n_", "pair_decay_"))]
        df[zero_cols] = df[zero_cols].fillna(0)

        # 7) Метки считаем ТОЛЬКО из будущего окна
        df = self._add_labels(df, fut)
        df["T"] = T
        return df

    def _pick_users(self, hist: pd.DataFrame, fut: pd.DataFrame) -> np.ndarray:
        both = np.intersect1d(hist["uid"].unique(), fut["uid"].unique())
        if len(both) > self.cfg.n_users:
            both = self.rng.choice(both, self.cfg.n_users, replace=False)
        return both

    def _pair_stats(self, hist_u: pd.DataFrame) -> pd.DataFrame:
        h = hist_u.copy()

        # Веса затухания w = exp(-age / tau) для каждого tau из конфига
        agg = {
            "pair_n_listens": ("ratio", "size"),          # сколько раз слушал
            "pair_n_plus": ("is_plus", "sum"),            # сколько раз дослушал >= 50%
            "pair_mean_ratio": ("ratio", "mean"),         # средний % прослушивания
            "days_since_last": ("age_days", "min"),       # давность последнего прослушивания
        }
        for tau in self.cfg.tau_days:
            col = f"pair_decay_{tau:g}d"
            h[col] = np.exp(-h["age_days"] / tau)
            agg[col] = (col, "sum")                       # сумма весов = "свежесть" интереса

        return h.groupby(["uid", "item_id"]).agg(**agg).reset_index()

    def _candidates(self, pair: pd.DataFrame, hist: pd.DataFrame, users: np.ndarray) -> pd.DataFrame:
        c = self.cfg

        # Источник 1: топ-N треков из личной истории (по самому длинному tau)
        sort_col = f"pair_decay_{max(c.tau_days):g}d"
        from_hist = (pair.sort_values(sort_col, ascending=False)
                         .groupby("uid").head(c.n_hist_cands)[["uid", "item_id"]])
        from_hist["src_hist"] = 1

        # Источник 2: глобально популярные за 7 дней (по ВСЕМ пользователям, но до T)
        pop_items = (hist[hist["age_days"] <= 7].groupby("item_id").size()
                         .nlargest(c.n_pop_cands).index.to_numpy())
        # Декартово произведение: каждый юзер получает каждый популярный трек
        from_pop = pd.MultiIndex.from_product([users, pop_items],
                                              names=["uid", "item_id"]).to_frame(index=False)
        from_pop["src_pop"] = 1

        # Объединяем; если пара пришла из двух источников, оба флага = 1
        cands = pd.concat([from_hist, from_pop], ignore_index=True).fillna(0)
        return cands.groupby(["uid", "item_id"], as_index=False)[["src_hist", "src_pop"]].max()

    def _user_feats(self, hist_u: pd.DataFrame) -> pd.DataFrame:
        return hist_u.groupby("uid").agg(
            u_n_listens=("ratio", "size"),               # активность
            u_mean_ratio=("ratio", "mean"),              # скиппер или дослушивает
            u_organic_share=("is_organic", "mean"),      # сам ищет музыку или идёт по рекомендациям
            u_n_items=("item_id", "nunique"),            # разнообразие
        ).reset_index()

    def _item_feats(self, hist: pd.DataFrame, likes_h: pd.DataFrame) -> pd.DataFrame:
        # Базовая статистика трека по ВСЕМ пользователям (до T)
        base = hist.groupby("item_id").agg(
            i_pop_total=("ratio", "size"),
            i_mean_ratio=("ratio", "mean"),              # хорошие треки реже скипают
            i_plus_rate=("is_plus", "mean"),
            i_n_users=("uid", "nunique"),
            i_length=("track_length_seconds", "mean"),
        )
        # Популярность в коротких окнах: "хит сейчас" против "хит вообще"
        base["i_pop_1d"] = hist[hist["age_days"] <= 1].groupby("item_id").size()
        base["i_pop_7d"] = hist[hist["age_days"] <= 7].groupby("item_id").size()
        # Лайки трека и like-rate = лайки / прослушивания
        base["i_n_likes"] = likes_h.groupby("item_id").size()
        base[["i_pop_1d", "i_pop_7d", "i_n_likes"]] = base[["i_pop_1d", "i_pop_7d", "i_n_likes"]].fillna(0)
        base["i_like_rate"] = base["i_n_likes"] / base["i_pop_total"]
        return base.reset_index()

    def _add_labels(self, df: pd.DataFrame, fut: pd.DataFrame) -> pd.DataFrame:
        # Позитивы: пары (юзер, трек), дослушанные >= порога в целевом окне
        pos = (fut[fut["played_ratio_pct"] >= self.cfg.pos_threshold][["uid", "item_id"]]
                  .drop_duplicates())
        pos["y"] = 1

        # Сколько всего позитивов у юзера в окне (включая те, что НЕ попали в кандидаты).
        # Нужно, чтобы Recall@k честно считался от всех позитивов, а не только от кандидатов.
        n_pos = pos.groupby("uid").size().rename("n_pos_total")

        df = df.merge(pos, on=["uid", "item_id"], how="left")
        df["y"] = df["y"].fillna(0).astype(int)
        df = df.merge(n_pos, on="uid", how="left")
        df["n_pos_total"] = df["n_pos_total"].fillna(0).astype(int)
        return df

